# 07 · Full-raster inference and submission (Milestone 5D)

Loads the registry **champion** (status `promoted`), averages its CV fold models over the full footprint, applies the area-budget export policy (DTI is maximised by a binary map at the cross-fitted budget) and writes one validated single-band float32 GeoTIFF per known-fault policy into `submissions/` with a provenance JSON. Uploading and the public score are **manual**: record each upload in `reports/leaderboard_log.csv` (never mixed with local scores).

In [ ]:
# --- session bootstrap (idempotent; identical in notebooks 00-07) ---------------------------------
REPO = 'mybenkhadda/GEMS---drivendata'
COMMIT = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/GEMS_2'
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.exists('/content/GEMS/.git'):
    subprocess.run(['git', 'clone', '-q', url, '/content/GEMS'], check=True)
subprocess.run(['git', '-C', '/content/GEMS', 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', '/content/GEMS', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/GEMS[colab]'], check=True)
if '/content/GEMS' not in sys.path:
    sys.path.insert(0, '/content/GEMS')
from gems_bench.runtime import colab
colab.configure(drive_root=DRIVE_ROOT)
colab.stage_core(); colab.stage_legacy(checkpoints=True);

In [ ]:
from gems_bench.experiment import Experiment
from gems_bench.artifacts import ModelRegistry
C = '/content/GEMS/configs/experiments/'
s = Experiment.from_yaml(C + 'z_submission.yaml')
display(ModelRegistry(s.paths.registry).table())
s.run_all()
s.rd.load_table('submission_files')

In [ ]:
# Leaderboard log (manual). Append a row after each upload; scoreboards read it into a separate column.
import pandas as pd
from pathlib import Path
from gems_bench.evaluation.scoreboard import LEADERBOARD_COLUMNS
log = Path(DRIVE_ROOT) / 'reports' / 'leaderboard_log.csv'
if not log.exists():
    pd.DataFrame(columns=LEADERBOARD_COLUMNS).to_csv(log, index=False)
pd.read_csv(log)